In [2]:
# Thư viện WebSocket để kết nối real-time với Bitget API
import websocket

# Thư viện xử lý JSON (dữ liệu từ API Bitget ở định dạng JSON)
import json

# Thư viện ghi file CSV để lưu dữ liệu ticker và candlestick
import csv

# Thư viện xử lý thời gian (timestamp, datetime conversion)
import datetime

# Thư viện xử lý file và thư mục (kiểm tra file tồn tại, kích thước file)
import os

# Thư viện pandas để xử lý dữ liệu dạng bảng (chưa sử dụng trong code này)
import pandas as pd

# Thư viện threading để chạy nhiều tác vụ cùng lúc (WebSocket chạy trong background)
import threading

# Thư viện time để tạm dừng chương trình và điều khiển thời gian chạy
import time

# Thư viện SQLAlchemy để kết nối database (đã import nhưng chưa sử dụng)
from sqlalchemy import create_engine

# Thư viện xử lý URL encoding (chưa sử dụng trong code này)
import urllib.parse

In [3]:
# ==================== CẤU HÌNH KẾT NỐI VÀ DỮ LIỆU ====================

# URL WebSocket public của Bitget để nhận dữ liệu real-time
WEBSOCKET_URL = "wss://ws.bitget.com/v2/ws/public"

# Danh sách các cặp cryptocurrency cần theo dõi (SOL, BTC, ETH với USDT)
# INSTRUMENT_IDS = ["SOLUSDT", "BTCUSDT", "ETHUSDT"]  

# Tên file CSV để lưu dữ liệu ticker (giá hiện tại, volume, thay đổi 24h)
CSV_FILE_NAME = "data.csv"


# TRY CẬP VÀO products ĐỂ LẤY DANH SÁCH COIN

In [7]:
import requests
url = "https://api.bitget.com/api/v2/spot/public/symbols"

try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    data = response.json()

    coin_list = data.get('data')

    if coin_list is not None:
        coin_names = [coin['symbol'] for coin in coin_list]
          # In 10 coin đầu
    else:
        print(" Không có trường 'data' trong phản hồi.")
except requests.exceptions.RequestException as e:
    print(f" Lỗi khi gọi API: {e}")
coin_tuy_y = coin_names  # Lấy 10 coin đầu tiên
INSTRUMENT_IDS = [f"{coin}" for coin in coin_tuy_y]  # Tạo danh sách cặp giao dịch với USDT
print(INSTRUMENT_IDS)



['LUMIAUSDT', 'GOATUSDT', 'ENAEUR', 'TONEUR', 'SXRPSUSDT', 'SHIBEUR', 'WLDEUR', 'DBRUSDT', 'ACTUSDT', 'GRASSUSDT', 'NEIROCTOEUR', 'SUIEUR', 'TRXUSDT', 'XUSDT', 'LINKUSDT', 'ONDOEUR', 'UNIUSDT', 'SUSHIUSDT', 'COMPUSDT', 'SCREUR', 'AAVEUSDT', 'SCRUSDT', 'YFIUSDT', 'NEAREUR', 'DOGEUSDT', 'CHZUSDT', 'PNUTUSDT', 'SWELLUSDT', 'WEETHETH', 'LUCEUSDT', 'FUSEUSDT', 'BANUSDT', 'NFPUSDT', 'NSUSDT', 'HIPPOUSDT', 'PEPECOINUSDT', 'FARTCOINUSDT', 'PEAQUSDT', 'MVUSDT', 'BTCWUSD', 'MOBILEUSDT', 'SWCHUSDT', 'AIUSDT', 'SOLUSDE', 'SNEKUSDT', 'RIFSOLUSDT', 'OORTUSDT', 'SOLOUSDT', 'XAIUSDT', 'FISUSDT', 'ELAUSDT', 'BONEUSDT', 'RDNTUSDT', 'SAROSUSDT', 'MANTAUSDT', 'PONKEUSDT', 'PROPSUSDT', 'UPCUSDT', 'ORAIUSDT', 'IOSTUSDT', 'COREUSDT', 'ALTUSDT', 'ONDOUSDT', 'WIFUSDT', 'AVAUSDT', 'THEUSDT', 'CHESSUSDT', 'ORNJUSDT', 'JUPUSDT', 'MCHUSDT', 'DMAILUSDT', 'BLURUSDT', 'API3USDT', 'XAUTUSDT', 'RAREUSDT', 'TRUUSDT', 'BIZAUSDT', 'USDCEUR', 'VRUSDT', 'FIREUSDT', 'CFXUSDT', 'ZETAUSDT', 'MDTUSDT', 'COREUMUSDT', 'VELODROMEU

# CẤU HÌNH HOÀN CHỈNH CHO TICKER + CANDLESTICK

In [5]:


# URL WebSocket public của Bitget để nhận dữ liệu real-time
WEBSOCKET_URL = "wss://ws.bitget.com/v2/ws/public"

# Danh sách các cặp cryptocurrency cần theo dõi
# INSTRUMENT_IDS = [coin_names]  # Lấy danh sách coin từ API

# File CSV lưu dữ liệu ticker (giá hiện tại, volume, bid/ask)
CSV_FILE_NAME = "data.csv"

# File CSV lưu dữ liệu candlestick (nến OHLC)
CANDLESTICK_CSV_FILE = "candlestick_data.csv"

# Các khung thời gian candlestick cần theo dõi
TIMEFRAMES = ["candle1m", "candle5m", "candle15m", "candle1H"]

# HÀM TẠO FILE CSV CHO DỮ LIỆU TICKER

In [6]:


def tao_file_csv():
    """Tạo file CSV với header chuẩn cho dữ liệu ticker"""
    
    # Định nghĩa các cột dữ liệu cho ticker (giá, volume, thống kê 24h)
    header = [
    "thoi_gian" ,          # Thời gian hệ thống ghi nhận dữ liệu (local time)
    "timestamp_api",       # Timestamp từ API Bitget (ms)
    "instId",              # Mã sản phẩm (ví dụ: ETHUSDT, BTCUSDT)
    "lastPr",              # Giá giao dịch gần nhất (current price)
    "bidPr",               # Giá mua cao nhất (highest bid)
    "askPr",               # Giá bán thấp nhất (lowest ask)
    "bidSz",               # Khối lượng đặt mua tại giá bid
    "askSz",               # Khối lượng đặt bán tại giá ask
    "open24h",             # Giá mở cửa 24 giờ trước
    "high24h",             # Giá cao nhất trong 24h
    "low24h",              # Giá thấp nhất trong 24h
    "change24h",           # Phần trăm thay đổi trong 24h
    "baseVolume",          # Khối lượng giao dịch theo base currency (coin)
    "quoteVolume",         # Khối lượng giao dịch theo quote currency (USDT)
    "openUtc",             # Giá mở cửa tại UTC+0
    "changeUtc24h",        # Thay đổi giá tại UTC+0 trong 24h
    "action"               # Loại dữ liệu: snapshot (cũ) hoặc update (mới)
]

    # Kiểm tra file có tồn tại và có dữ liệu chưa
    if not os.path.exists(CSV_FILE_NAME) or os.path.getsize(CSV_FILE_NAME) == 0:
        # Tạo file mới với header nếu chưa tồn tại
        with open(CSV_FILE_NAME, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(header)
        print(f"Đã tạo file CSV với {len(header)} cột: {CSV_FILE_NAME}")
    else:
        print(f"File CSV đã tồn tại: {CSV_FILE_NAME}")

# Gọi hàm tạo file ticker ngay khi chạy
tao_file_csv()

File CSV đã tồn tại: data.csv


# HÀM TẠO FILE CSV CHO DỮ LIỆU CANDLESTICK

In [7]:


def tao_file_candlestick_csv():
    """Tạo file CSV với header chuẩn cho dữ liệu candlestick (nến OHLC)"""
    
    # Định nghĩa các cột dữ liệu cho candlestick (nến OHLC + volume)
    candlestick_header = [
        "thoi_gian_he_thong",      # Thời gian hệ thống ghi nhận (local time)
    "start_time",              # Thời gian bắt đầu nến (từ API, đã convert sang readable)
    "open_price",              # Giá mở cửa của nến (Open)
    "high_price",              # Giá cao nhất của nến (High)
    "low_price",               # Giá thấp nhất của nến (Low)
    "close_price",             # Giá đóng cửa của nến (Close)
    "volume_base",             # Khối lượng giao dịch theo base currency
    "volume_quote",            # Khối lượng giao dịch theo quote currency
    "volume_usdt",             # Khối lượng giao dịch tính bằng USDT
    "instId",                  # Mã sản phẩm (ETHUSDT, BTCUSDT, SOLUSDT)
    "timeframe",               # Khung thời gian nến (1m, 5m, 15m, 1H)
    "action"                   # Loại dữ liệu: snapshot hoặc update
]
    
    # Kiểm tra file candlestick có tồn tại chưa
    if not os.path.exists(CANDLESTICK_CSV_FILE) or os.path.getsize(CANDLESTICK_CSV_FILE) == 0:
        # Tạo file mới với header nếu chưa tồn tại
        with open(CANDLESTICK_CSV_FILE, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(candlestick_header)
        print(f"Đã tạo file Candlestick CSV với {len(candlestick_header)} cột: {CANDLESTICK_CSV_FILE}")
    else:
        print(f"File Candlestick CSV đã tồn tại: {CANDLESTICK_CSV_FILE}")

# Gọi hàm tạo file candlestick ngay khi chạy
tao_file_candlestick_csv()

File Candlestick CSV đã tồn tại: candlestick_data.csv


# BIẾN TOÀN CỤC VÀ HÀM XỬ LÝ WEBSOCKET

In [8]:


# Mảng lưu trữ toàn bộ dữ liệu ticker và candlestick để debug
all_ticker_data = []
all_candlestick_data = []

def on_open_enhanced(ws):
    """Hàm được gọi khi WebSocket kết nối thành công"""
    print("Đã kết nối thành công")
    
    # === ĐĂNG KÝ THEO DÕI TICKER CHO TẤT CẢ SYMBOLS ===
    for inst_id in INSTRUMENT_IDS:
        ticker_message = {
            "op": "subscribe",        # Lệnh đăng ký
            "args": [
                {    
                    "instType": "SPOT",      # Loại sản phẩm: giao dịch spot
                    "channel": "ticker",     # Channel ticker (giá hiện tại)
                    "instId": inst_id        # Symbol cần theo dõi
                }
            ]
        }
        ws.send(json.dumps(ticker_message))
        print(f"Đang theo dõi ticker {inst_id}")
    
    # === ĐĂNG KÝ THEO DÕI CANDLESTICK CHO TẤT CẢ SYMBOLS VÀ TIMEFRAMES ===
    for inst_id in INSTRUMENT_IDS:
        for timeframe in TIMEFRAMES:
            candlestick_message = {
                "op": "subscribe",
                "args": [
                    {    
                        "instType": "SPOT",      # Loại sản phẩm: spot
                        "channel": timeframe,    # Channel candlestick (candle1m, candle5m, ...)
                        "instId": inst_id        # Symbol cần theo dõi
                    }
                ]
            }
            ws.send(json.dumps(candlestick_message))
            print(f"Đang theo dõi candlestick {inst_id} - {timeframe}")
    
    print(f"Hoàn tất subscribe cho {len(INSTRUMENT_IDS)} coins với {len(TIMEFRAMES)} timeframes")

def on_message_enhanced(ws, message_str):
    """Hàm xử lý tin nhắn từ WebSocket - route dữ liệu đến đúng processor"""
    global all_ticker_data, all_candlestick_data
    
    # Parse JSON từ message
    data = json.loads(message_str)
    
    # Kiểm tra loại channel để route dữ liệu đúng chỗ
    if "arg" in data and "channel" in data["arg"]:
        channel = data["arg"]["channel"]
        
        if channel == "ticker":
            # Dữ liệu ticker: giá hiện tại, volume, bid/ask
            all_ticker_data.append(data)
            process_ticker_data(data)
            
        elif channel.startswith("candle"):
            # Dữ liệu candlestick: nến OHLC với các timeframe khác nhau
            all_candlestick_data.append(data)
            process_candlestick_data(data)

def process_ticker_data(data):
    """Xử lý và lưu dữ liệu ticker vào CSV"""
    if "data" in data and data["data"]:
        ticker = data["data"][0]  # Lấy ticker đầu tiên trong array
        
        # === TRÍCH XUẤT DỮ LIỆU TICKER ===
        thoi_gian = datetime.datetime.now().isoformat()  # Thời gian hệ thống ghi nhận
        instId = ticker.get('instId')                   # Mã sản phẩm (ETHUSDT, BTCUSDT)
        gia = ticker.get('lastPr')                      # Giá giao dịch gần nhất
        gia_mua = ticker.get('bidPr')                   # Giá mua cao nhất (bid)
        gia_ban = ticker.get('askPr')                   # Giá bán thấp nhất (ask)
        khoi_luong_24h = ticker.get('baseVolume')       # Khối lượng 24h theo base currency
        high24h = ticker.get('high24h')                 # Giá cao nhất 24h
        low24h = ticker.get('low24h')                   # Giá thấp nhất 24h
        best_purchase_price = ticker.get('bidSz')       # Khối lượng đặt mua tại bid
        best_sale_price = ticker.get('askSz')           # Khối lượng đặt bán tại ask
        change24h = ticker.get('change24h')             # % thay đổi 24h
        base_volume = ticker.get('baseVolume')          # Khối lượng base (trùng khoi_luong_24h)
        quote_volume = ticker.get('quoteVolume')        # Khối lượng quote (USDT)
        open_utc = ticker.get('openUtc')                # Giá mở UTC+0
        open24h = ticker.get('open24h')                 # Giá mở 24h trước
        timestamp_api = ticker.get('ts')                # Timestamp từ API
        action = data.get('action', 'unknown')          # snapshot/update/unknown nếu không có
                
        # === HIỂN THỊ THÔNG TIN TICKER TRÊN CONSOLE ===
        print(f"TICKER {instId} | Giá: {gia} | Thay đổi 24h: {change24h}")
        
        # === GHI DỮ LIỆU VÀO FILE CSV TICKER ===
        with open(CSV_FILE_NAME, mode='a', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow([
    thoi_gian, gia, gia_mua, gia_ban, khoi_luong_24h, high24h, low24h, instId,
    best_purchase_price, best_sale_price, change24h,
    base_volume, quote_volume, open_utc, open24h,
    timestamp_api, action
])


def process_candlestick_data(data):
    """Xử lý và lưu dữ liệu candlestick (nến OHLC) vào CSV"""
    if "data" in data and data["data"]:
        candle = data["data"][0]  # Lấy nến đầu tiên trong array
        arg = data.get("arg", {})
        
        # === THỜI GIAN HỆ THỐNG ===
        thoi_gian_he_thong = datetime.datetime.now().isoformat()
        
        # === TRÍCH XUẤT DỮ LIỆU TỪ CANDLE ARRAY (FORMAT CỦA BITGET) ===
        start_time = candle[0]  # Timestamp bắt đầu nến (ms)
        open_price = candle[1]   # Giá mở cửa (Open)
        high_price = candle[2]   # Giá cao nhất (High)
        low_price = candle[3]    # Giá thấp nhất (Low)
        close_price = candle[4]  # Giá đóng cửa (Close)
        volume_base = candle[5]  # Khối lượng base currency
        volume_quote = candle[6] # Khối lượng quote currency
        volume_usdt = candle[7]  # Khối lượng tính bằng USDT
        
        # === THÔNG TIN METADATA ===
        instId = arg.get('instId')      # Symbol (ETHUSDT, BTCUSDT)
        timeframe = arg.get('channel')   # Timeframe (candle1m, candle5m)
        action = data.get('action', 'unknown')  # snapshot/update
        
        # === CONVERT TIMESTAMP SANG ĐỊNH DẠNG DỄ ĐỌC ===
        try:
            # Convert từ milliseconds sang datetime readable
            start_time_readable = datetime.datetime.fromtimestamp(int(start_time)/1000).isoformat()
        except:
            # Nếu convert lỗi thì giữ nguyên
            start_time_readable = start_time
        
        # === HIỂN THỊ THÔNG TIN CANDLESTICK TRÊN CONSOLE ===
        print(f"CANDLE {instId} {timeframe} | O:{open_price} H:{high_price} L:{low_price} C:{close_price}")
        
       # === GHI DỮ LIỆU VÀO FILE CSV CANDLESTICK ===
        with open(CANDLESTICK_CSV_FILE, mode='a', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow([
                thoi_gian_he_thong,  # Thời gian hệ thống
                start_time_readable,  # Thời gian bắt đầu nến (đã convert)
                open_price,          # Giá mở cửa
                high_price,          # Giá cao nhất  
                low_price,           # Giá thấp nhất
                close_price,         # Giá đóng cửa
                volume_base,         # Khối lượng base
                volume_quote,        # Khối lượng quote
                volume_usdt,         # Khối lượng USDT
                instId,              # Sản phẩm (ETHUSDT)
                timeframe,           # Timeframe (candle1m, candle5m, ...)
                action               # Action (snapshot/update)
            ])

def on_error(ws, error):
    """Hàm xử lý lỗi WebSocket"""
    print(f"Lỗi: {error}")

def on_close(ws, close_status_code, close_msg):
    """Hàm xử lý khi WebSocket đóng kết nối"""
    print(f"Kết nối đã đóng")

# CHẠY WEBSOCKET VÀ ĐIỀU KHIỂN CHƯƠNG TRÌNH

In [9]:

# === CẤU HÌNH THỜI GIAN CHẠY ===
a = 60  # 1 phút = 60 giây
b = a * 60  # 1 giờ = 3600 giây
c = b * 24  # 1 ngày = 86400 giây

def run_ws_enhanced():
    """Hàm chạy WebSocket với ping để duy trì kết nối"""
    # Chạy WebSocket với ping mỗi 30 giây, timeout 10 giây
    ws.run_forever(ping_interval=30, ping_timeout=10)

# === TẠO ĐỐI TƯỢNG WEBSOCKET VỚI CÁC CALLBACK FUNCTIONS ===
ws = websocket.WebSocketApp(WEBSOCKET_URL,
                          on_open=on_open_enhanced,      # Gọi khi kết nối thành công
                          on_message=on_message_enhanced,  # Gọi khi nhận tin nhắn
                          on_error=on_error,            # Gọi khi có lỗi
                          on_close=on_close)            # Gọi khi đóng kết nối

# === THÔNG BÁO BẮT ĐẦU CRAWLING ===
print("Bắt đầu kết nối đến Bitget (Ticker + Candlestick)...")
print(f"Ticker data sẽ được lưu vào: {CSV_FILE_NAME}")
print(f"Candlestick data sẽ được lưu vào: {CANDLESTICK_CSV_FILE}")
print("Nhấn Ctrl+C để dừng")

# === CHẠY WEBSOCKET TRONG BACKGROUND THREAD ===
ws_thread = threading.Thread(target=run_ws_enhanced)
ws_thread.daemon = True  # Thread sẽ tự động tắt khi main thread tắt
ws_thread.start()

# === THIẾT LẬP THỜI GIAN CHẠY ===
run_duration = a*6 # Chạy trong 1 phút (60 giây)

# === CHẠY CHƯƠNG TRÌNH VÀ XỬ LÝ KEYBOARD INTERRUPT ===
try:
    time.sleep(run_duration)  # Sleep trong thời gian đã định
except KeyboardInterrupt:
    print("\nĐã dừng bằng Ctrl+C")  # Xử lý khi user nhấn Ctrl+C

# === ĐÓNG KẾT NỐI VÀ THỐNG KÊ ===
ws.close()
print(f"Đã ngắt kết nối sau {run_duration} giây.")
print(f"Ticker data: {len(all_ticker_data)} messages")        # Số tin nhắn ticker
print(f"Candlestick data: {len(all_candlestick_data)} messages")  # Số tin nhắn candlestick

Bắt đầu kết nối đến Bitget (Ticker + Candlestick)...
Ticker data sẽ được lưu vào: data.csv
Candlestick data sẽ được lưu vào: candlestick_data.csv
Nhấn Ctrl+C để dừng
Đã kết nối thành công
Đang theo dõi ticker U2U
Đang theo dõi ticker WUF
Đang theo dõi ticker CATDOG
Đang theo dõi ticker INVITE
Đang theo dõi ticker NEIROETH
Đang theo dõi ticker ORDER
Đang theo dõi ticker SWELL
Đang theo dõi ticker BLASTUP
Đang theo dõi ticker WHY
Đang theo dõi ticker ZKLITEETH
Đang theo dõi ticker SUNDOG
Đang theo dõi ticker Z
Đang theo dõi ticker CEC
Đang theo dõi ticker BTC
Đang theo dõi ticker DLC
Đang theo dõi ticker USDT
Đang theo dõi ticker OGLG
Đang theo dõi ticker ETH
Đang theo dõi ticker DOGS
Đang theo dõi ticker IOST
Đang theo dõi ticker GEEK
Đang theo dõi ticker LTC
Đang theo dõi ticker BCH
Đang theo dõi ticker QTUM
Đang theo dõi ticker GINNAN
Đang theo dõi ticker ETC
Đang theo dõi ticker SOLS
Đang theo dõi ticker MAK
Đang theo dõi ticker AKT
Đang theo dõi ticker BOX
Đang theo dõi ticker SNIFT

: 

: 